# ONNX／TensorRT

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/20-deployment/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.6.1'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7', 'onnx==1.19.1', 'onnxruntime==1.23.2']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""Real ONNX export, checker, ORT CPU execution and decoded-box parity."""
import json
import statistics
import time
from pathlib import Path
import numpy as np
import onnx
import onnxruntime as ort
from onnxruntime.capi.onnxruntime_pybind11_state import InvalidArgument
import torch
from miniyolo.models import GridDetector
from miniyolo.geometry import letterbox, undo_letterbox
from miniyolo.inference import decode_grid
from miniyolo.targets import build_targets
from miniyolo.losses import grid_loss
from miniyolo.data import ShapeDataset, collate


def preprocess(rgb):
    assert rgb.dtype == np.uint8 and rgb.ndim == 3 and rgb.shape[-1] == 3
    image = torch.from_numpy(rgb.copy()).permute(2, 0, 1).float() / 255
    image, _, metadata = letterbox(image, torch.empty(0, 4), size=64)
    return image, metadata


def restore(raw, metadata):
    result = decode_grid(torch.from_numpy(raw.copy()), image_size=64, score_threshold=.05, nms_iou=.5)
    for pred, meta in zip(result, metadata):
        pred['boxes'] = undo_letterbox(pred['boxes'], meta)
    return result


def median_ms(function, iterations=20):
    for _ in range(3):
        function()
    times = []
    for _ in range(iterations):
        start = time.perf_counter()
        function()
        times.append((time.perf_counter() - start) * 1000)
    return statistics.median(times)


def paired_median_ms(function_a, function_b, rounds):
    """Medians (ms) of a, of b and of the per-round differences a - b; each round runs both in alternating order."""
    for _ in range(3):
        function_a()
        function_b()
    times_a, times_b = [], []
    for i in range(rounds):
        order = [(function_a, times_a), (function_b, times_b)]
        if i % 2 == 1:
            order.reverse()
        for function, times in order:
            start = time.perf_counter()
            function()
            times.append((time.perf_counter() - start) * 1000)
    differences = [a - b for a, b in zip(times_a, times_b)]
    return statistics.median(times_a), statistics.median(times_b), statistics.median(differences)


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    output = Path('artifacts/lesson-20')
    output.mkdir(parents=True, exist_ok=True)
    model = GridDetector(width=8)
    # A real forward/backward/step supplies locally learned weights; no pretrained download.
    data = ShapeDataset(n=4, size=64, seed=5100)
    images, targets = collate([data[i] for i in range(4)])
    optimizer = torch.optim.SGD(model.parameters(), lr=.01)
    optimizer.zero_grad()
    grid_loss(model(images), build_targets(targets))['total'].backward()
    optimizer.step()
    model.eval()
    onnx_path = output / 'grid.onnx'
    torch.onnx.export(model, images[:1], str(onnx_path), input_names=['images'], output_names=['raw_grid'],
                      dynamic_axes={'images': {0: 'batch'}, 'raw_grid': {0: 'batch'}},
                      opset_version=17, dynamo=False)
    onnx.checker.check_model(onnx.load(str(onnx_path)))
    options = ort.SessionOptions()
    options.intra_op_num_threads, options.inter_op_num_threads = 2, 1
    session = ort.InferenceSession(str(onnx_path), sess_options=options, providers=['CPUExecutionProvider'])
    assert session.get_inputs()[0].shape == ['batch', 3, 64, 64]
    rng = np.random.default_rng(7)
    sources = [rng.integers(0, 256, shape, dtype=np.uint8)
               for shape in ((64, 96, 3), (48, 80, 3), (80, 48, 3))]
    processed = [preprocess(rgb) for rgb in sources]
    batch = torch.stack([item[0] for item in processed])
    metadata = [item[1] for item in processed]
    errors, box_counts = [], []
    batches_checked = [1, 2, 3]
    with torch.no_grad():
        for b in batches_checked:
            assert batch[:b].shape == (b, 3, 64, 64)
            torch_raw = model(batch[:b]).numpy()
            ort_raw = session.run(['raw_grid'], {'images': batch[:b].numpy()})[0]
            assert torch_raw.shape == ort_raw.shape == (b, 4, 4, 7)
            error = float(np.abs(torch_raw - ort_raw).max())
            errors.append(error)
            np.testing.assert_allclose(torch_raw, ort_raw, rtol=1e-5, atol=1e-5)
            pytorch_predictions, ort_predictions = restore(torch_raw, metadata[:b]), restore(ort_raw, metadata[:b])
            for a, o in zip(pytorch_predictions, ort_predictions):
                torch.testing.assert_close(a['boxes'], o['boxes'], rtol=1e-5, atol=1e-4)
                torch.testing.assert_close(a['scores'], o['scores'], rtol=1e-5, atol=1e-6)
                assert torch.equal(a['labels'], o['labels'])
            # Two empty results also pass the comparison above, so each source image in this batch must yield boxes.
            box_counts = [len(a['boxes']) for a in pytorch_predictions]
            assert all(c > 0 for c in box_counts), f'B={b}: a source has no restored box {box_counts}'
    try:
        session.run(['raw_grid'], {'images': np.zeros((1, 3, 80, 80), dtype=np.float32)})
    except InvalidArgument:
        spatial_rejected = True
    else:
        spatial_rejected = False
    assert spatial_rejected, 'Export is intentionally dynamic-batch, fixed spatial size.'
    with torch.no_grad():
        pytorch_ms = median_ms(lambda: model(batch[:1]))
    ort_ms = median_ms(lambda: session.run(['raw_grid'], {'images': batch[:1].numpy()}))
    def ort_pipeline():
        image, meta = preprocess(sources[0])
        raw = session.run(['raw_grid'], {'images': image[None].numpy()})[0]
        return restore(raw, [meta])
    def torch_pipeline():
        image, meta = preprocess(sources[0])
        with torch.no_grad():
            raw = model(image[None]).numpy()
        return restore(raw, [meta])
    # Both pipelines run in every round, alternating which goes first, so a slow drift of the machine
    # (e.g. clock speed) lands on both sides of each per-round difference instead of on one whole series.
    end_to_end_rounds = 40
    torch_end_to_end_ms, ort_end_to_end_ms, end_to_end_difference_ms = paired_median_ms(
        torch_pipeline, ort_pipeline, end_to_end_rounds)
    batch2_ms = median_ms(lambda: session.run(['raw_grid'], {'images': batch[:2].numpy()}))
    report = {'onnx': str(onnx_path), 'opset': 17, 'providers': session.get_providers(),
              'input_shape': session.get_inputs()[0].shape, 'batches_checked': batches_checked, 'max_abs_raw_errors': errors,
              'decoded_boxes_scores_labels_match': True, 'restored_boxes_per_source': box_counts,
              'dynamic_spatial': False, 'spatial80_rejected': spatial_rejected,
              'median_ms': {'torch_raw_batch1': pytorch_ms, 'ort_raw_batch1': ort_ms,
                            'torch_preprocess_to_restored_boxes': torch_end_to_end_ms,
                            'ort_preprocess_to_restored_boxes': ort_end_to_end_ms,
                            'paired_torch_minus_ort_preprocess_to_restored_boxes': end_to_end_difference_ms,
                            'ort_raw_batch2': batch2_ms},
              'end_to_end_paired_rounds': end_to_end_rounds,
              'ort_raw_batch2_images_per_second': 2000 / batch2_ms,
              'versions': {'torch': torch.__version__, 'onnx': onnx.__version__, 'onnxruntime': ort.__version__},
              'limits': 'CPU float32 parity; one training step is not detection-quality evidence; this CPU case does not run TensorRT/GPU; separate L4 evidence is documented'}
    (output / 'report.json').write_text(json.dumps(report, indent=2) + '\n')
    print(json.dumps(report, indent=2))
    print('real ONNX export + checker + ORT CPU + restored-box parity completed')


if __name__ == '__main__':
    main()


{
  "onnx": "artifacts/lesson-20/grid.onnx",
  "opset": 17,
  "providers": [
    "CPUExecutionProvider"
  ],
  "input_shape": [
    "batch",
    3,
    64,
    64
  ],
  "batches_checked": [
    1,
    2,
    3
  ],
  "max_abs_raw_errors": [
    4.76837158203125e-07,
    4.76837158203125e-07,
    4.76837158203125e-07
  ],
  "decoded_boxes_scores_labels_match": true,
  "restored_boxes_per_source": [
    16,
    16,
    16
  ],
  "dynamic_spatial": false,
  "spatial80_rejected": true,
  "median_ms": {
    "torch_raw_batch1": 0.1509749990873388,
    "ort_raw_batch1": 0.036599500162992626,
    "torch_preprocess_to_restored_boxes": 2.3252515002241125,
    "ort_preprocess_to_restored_boxes": 2.1557750005740672,
    "paired_torch_minus_ort_preprocess_to_restored_boxes": 0.17311249939666595,
    "ort_raw_batch2": 0.04289900061849039
  },
  "end_to_end_paired_rounds": 40,
  "ort_raw_batch2_images_per_second": 46621.132687598256,
  "versions": {
    "torch": "2.9.1+cpu",
    "onnx": "1.19.1",
  